# ShopMind — Model Training (BU SCC)
**Run this notebook on BU SCC with a GPU node.**

What this notebook trains:
1. **ABSA Model** — BERT fine-tuned for Aspect-Based Sentiment Analysis
2. **Fake Review Classifier** — XGBoost on linguistic + behavioral features

At the end, zip `outputs/` and download it to your Mac.

**Estimated time:** 2-4 hours on a V100/A100

## 0. Install Dependencies

In [1]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split())

pip('torch torchvision --index-url https://download.pytorch.org/whl/cu118')
pip('transformers==4.40.0')
pip('datasets==2.18.0')
pip('accelerate==0.29.0')
pip('evaluate==0.4.1')
pip('scikit-learn==1.4.2')
pip('xgboost==2.0.3')
pip('pandas numpy tqdm joblib')
print('✓ All packages installed')

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


✓ All packages installed



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 1. Setup & GPU Check

In [2]:
import os, json, gzip, requests, joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Output directories
os.makedirs('outputs/models/absa_model', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)
os.makedirs('data', exist_ok=True)
print('✓ Directories ready')

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA H200 NVL
VRAM: 150.1 GB
✓ Directories ready


## 2. Download Amazon Reviews Data

In [3]:
from datasets import load_dataset
import pandas as pd

MAX_REVIEWS = 100_000

print('Streaming Amazon Reviews 2023 (Electronics) from HuggingFace...')
print('This avoids downloading the full ~10GB file.')

ds = load_dataset(
    'McAuley-Lab/Amazon-Reviews-2023',
    'raw_review_Electronics',
    split='full',
    streaming=True,
    trust_remote_code=True
)

rows = []
for i, row in enumerate(ds):
    if i >= MAX_REVIEWS:
        break
    rows.append(row)
    if (i + 1) % 10000 == 0:
        print(f'  {i+1:,} reviews loaded...')

print(f'✓ Loaded {len(rows):,} reviews')

Streaming Amazon Reviews 2023 (Electronics) from HuggingFace...
This avoids downloading the full ~10GB file.


  10,000 reviews loaded...
  20,000 reviews loaded...
  30,000 reviews loaded...
  40,000 reviews loaded...
  50,000 reviews loaded...
  60,000 reviews loaded...
  70,000 reviews loaded...
  80,000 reviews loaded...
  90,000 reviews loaded...
  100,000 reviews loaded...
✓ Loaded 100,000 reviews


In [4]:
df = pd.DataFrame(rows)

# Rename columns to match project schema
rename = {
    'parent_asin': 'product_id',
    'text': 'reviewText',
    'rating': 'overall',
    'helpful_vote': 'helpful_votes'
}
df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

# Keep only needed columns (handle missing ones gracefully)
keep = ['product_id', 'user_id', 'overall', 'reviewText', 'verified_purchase', 'helpful_votes']
df = df[[c for c in keep if c in df.columns]]
df = df.dropna(subset=['product_id', 'reviewText'])
df['reviewText'] = df['reviewText'].astype(str)
df['overall'] = pd.to_numeric(df['overall'], errors='coerce')
df = df[df['reviewText'].str.len() >= 20].reset_index(drop=True)

print(f'✓ Clean reviews: {len(df):,}')
print(df['overall'].value_counts().sort_index())
df.to_csv('data/reviews.csv', index=False)
print('✓ Saved to data/reviews.csv')
df.head(3)

✓ Clean reviews: 92,487
overall
1.0     7685
2.0     4197
3.0     6850
4.0    14462
5.0    59293
Name: count, dtype: int64
✓ Saved to data/reviews.csv


,product_id,user_id,overall,reviewText,verified_purchase,helpful_votes
0,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,3.0,First & most offensive: they reek of gasoline ...,True,0
1,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1.0,These didn’t work. Idk if they were damaged in...,True,0
2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,5.0,I love these. They even come with a carry case...,True,0


## 3. Train ABSA Model (BERT Fine-Tuning)
Since we don't have labeled aspect data, we use **weak supervision**:
- Extract aspect keywords (battery, camera, screen, etc.) from reviews
- Use the review's star rating as the sentiment label for each aspect mentioned
- Fine-tune BERT as a 3-class classifier: Positive / Neutral / Negative

In [5]:
# ── Weak supervision: create aspect-sentiment pairs ──────────────────────────
ASPECT_KEYWORDS = {
    'battery':   ['battery', 'charge', 'charging', 'power', 'life'],
    'screen':    ['screen', 'display', 'resolution', 'brightness', 'pixel'],
    'camera':    ['camera', 'photo', 'picture', 'image', 'lens', 'video'],
    'sound':     ['sound', 'audio', 'speaker', 'volume', 'bass', 'noise'],
    'build':     ['build', 'quality', 'material', 'plastic', 'metal', 'durable'],
    'price':     ['price', 'value', 'worth', 'expensive', 'cheap', 'cost'],
    'shipping':  ['shipping', 'delivery', 'arrived', 'packaging', 'box'],
    'software':  ['app', 'software', 'update', 'interface', 'ui', 'bug'],
}

def rating_to_label(rating):
    if rating >= 4:  return 2  # positive
    elif rating == 3: return 1  # neutral
    else:             return 0  # negative

def extract_aspect_samples(df, max_samples=20000):
    samples = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting aspect samples'):
        text = str(row['reviewText']).lower()
        rating = row['overall']
        if pd.isna(rating):
            continue
        for aspect, keywords in ASPECT_KEYWORDS.items():
            if any(kw in text for kw in keywords):
                # Build input: "[aspect] [SEP] review text"
                input_text = f"{aspect} [SEP] {str(row['reviewText'])[:300]}"
                samples.append({'text': input_text, 'label': rating_to_label(rating), 'aspect': aspect})
                if len(samples) >= max_samples:
                    return samples
    return samples

samples = extract_aspect_samples(df)
absa_df = pd.DataFrame(samples)
print(f'✓ Created {len(absa_df):,} aspect-sentiment samples')
print(absa_df['label'].value_counts())
absa_df.head(3)

Extracting aspect samples:  16%|█▌        | 14624/92487 [00:00<00:02, 35539.88it/s]

✓ Created 20,000 aspect-sentiment samples
label
2    16137
0     2223
1     1640
Name: count, dtype: int64


,text,label,aspect
0,camera [SEP] First & most offensive: they reek...,1,camera
1,build [SEP] First & most offensive: they reek ...,1,build
2,price [SEP] First & most offensive: they reek ...,1,price


In [6]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
import evaluate

MODEL_NAME = 'distilbert-base-uncased'  # smaller + faster than BERT-base
NUM_LABELS = 3
ABSA_OUTPUT = 'outputs/models/absa_model'

# Train/val split
train_df, val_df = train_test_split(absa_df, test_size=0.1, random_state=42, stratify=absa_df['label'])
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_ds = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df[['text', 'label']]).map(tokenize, batched=True)
train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print('✓ Datasets tokenized')

/restricted/projectnb/dentalds/qwen_env_pkgs/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Train: 18,000 | Val: 2,000


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

✓ Datasets tokenized


In [7]:
# ── Fine-tune DistilBERT ──────────────────────────────────────────────────────
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1': f1_metric.compute(predictions=preds, references=labels, average='macro')['f1'],
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

args = TrainingArguments(
    output_dir=ABSA_OUTPUT,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=torch.cuda.is_available(),  # FP16 on GPU
    logging_steps=100,
    report_to='none',  # no wandb/mlflow needed on SCC
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('=== Starting ABSA training ===')
trainer.train()
print('✓ Training complete')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


=== Starting ABSA training ===


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.361900,0.305788,0.898500,0.687475
2,0.203200,0.225194,0.932500,0.817529
3,0.126000,0.184482,0.947500,0.864588


✓ Training complete


In [8]:
# Evaluate and save
metrics = trainer.evaluate()
print(f"\nFinal Val Accuracy: {metrics['eval_accuracy']:.4f}")
print(f"Final Val F1 (macro): {metrics['eval_f1']:.4f}")

# Save model + tokenizer
trainer.save_model(ABSA_OUTPUT)
tokenizer.save_pretrained(ABSA_OUTPUT)
print(f'✓ ABSA model saved to {ABSA_OUTPUT}/')


Final Val Accuracy: 0.9475
Final Val F1 (macro): 0.8646
✓ ABSA model saved to outputs/models/absa_model/


## 4. Train Fake Review Classifier (XGBoost)

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from xgboost import XGBClassifier
import re

# ── Feature Engineering ───────────────────────────────────────────────────────
def extract_features(df):
    feats = pd.DataFrame()
    text = df['reviewText'].fillna('').astype(str)

    # Linguistic features
    feats['review_length']    = text.str.len()
    feats['word_count']       = text.str.split().str.len()
    feats['avg_word_length']  = text.apply(lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0)
    feats['exclamation_count']= text.str.count('!')
    feats['question_count']   = text.str.count('\\?')
    feats['caps_ratio']       = text.apply(lambda t: sum(c.isupper() for c in t) / max(len(t), 1))
    feats['unique_word_ratio']= text.apply(lambda t: len(set(t.lower().split())) / max(len(t.split()), 1))
    feats['digit_count']      = text.str.count('\\d')
    feats['sentence_count']   = text.str.count('[.!?]')

    # Behavioral features
    feats['rating']           = pd.to_numeric(df['overall'], errors='coerce').fillna(3)
    feats['verified_purchase']= df['verified_purchase'].astype(int) if 'verified_purchase' in df.columns else 1
    feats['helpful_votes']    = pd.to_numeric(df.get('helpful_votes', 0), errors='coerce').fillna(0)

    # Rating deviation from product mean (suspicious = extreme outlier)
    prod_mean = df.groupby('product_id')['overall'].transform('mean')
    feats['rating_deviation'] = (feats['rating'] - prod_mean).abs()

    # Reviews per user (high count = suspicious)
    user_review_count = df.groupby('user_id')['user_id'].transform('count') if 'user_id' in df.columns else pd.Series(1, index=df.index)
    feats['user_review_count'] = user_review_count

    return feats.fillna(0)


# ── Weak labels (heuristic) ───────────────────────────────────────────────────
# Real proxy for "fake": extreme rating + very short + unverified + burst poster
def create_weak_labels(df, feats):
    fake_score = (
        ((feats['rating'] == 5) | (feats['rating'] == 1)).astype(int) * 1 +
        (feats['word_count'] < 10).astype(int) * 2 +
        (feats['verified_purchase'] == 0).astype(int) * 2 +
        (feats['user_review_count'] > 20).astype(int) * 2 +
        (feats['exclamation_count'] > 3).astype(int) * 1 +
        (feats['caps_ratio'] > 0.3).astype(int) * 1
    )
    # Label: 1 = fake (score >= 5), 0 = genuine
    labels = (fake_score >= 5).astype(int)
    print(f'Label distribution: {labels.value_counts().to_dict()}')
    return labels


feats = extract_features(df)
labels = create_weak_labels(df, feats)
print(f'✓ Features extracted: {feats.shape}')
feats.head(3)

Label distribution: {0: 79384, 1: 13103}
✓ Features extracted: (92487, 14)


,review_length,word_count,avg_word_length,exclamation_count,question_count,caps_ratio,unique_word_ratio,digit_count,sentence_count,rating,verified_purchase,helpful_votes,rating_deviation,user_review_count
0,1433,275,4.185455,1,0,0.019539,0.625455,10,16,3.0,1,0,0.000000,3
1,225,45,4.022222,0,0,0.017778,0.800000,0,4,1.0,1,0,0.000000,3
2,469,94,3.968085,2,0,0.021322,0.776596,0,8,5.0,1,0,1.136364,3


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    feats, labels, test_size=0.2, random_state=42, stratify=labels
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1,
    ))
])

print('=== Training XGBoost fake review classifier ===')
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(f'\nROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Genuine', 'Fake']))

FAKE_MODEL_PATH = 'outputs/models/fake_review_model.joblib'
joblib.dump(pipeline, FAKE_MODEL_PATH)
print(f'✓ Fake review model saved to {FAKE_MODEL_PATH}')

=== Training XGBoost fake review classifier ===

ROC-AUC: 1.0000
F1 Score: 0.9996

Classification Report:
              precision    recall  f1-score   support

     Genuine       1.00      1.00      1.00     15877
        Fake       1.00      1.00      1.00      2621

    accuracy                           1.00     18498
   macro avg       1.00      1.00      1.00     18498
weighted avg       1.00      1.00      1.00     18498

✓ Fake review model saved to outputs/models/fake_review_model.joblib


## 5. Save Feature Column Names (needed for local inference)

In [11]:
import json

# Save feature names so local inference uses same column order
meta = {
    'feature_columns': list(feats.columns),
    'absa_model_path': 'outputs/models/absa_model',
    'absa_base_model': 'distilbert-base-uncased',
    'absa_num_labels': 3,
    'label_map': {0: 'negative', 1: 'neutral', 2: 'positive'},
    'aspects': list(ASPECT_KEYWORDS.keys())
}
with open('outputs/models/training_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('✓ Saved training_meta.json')

✓ Saved training_meta.json


## 6. Package Outputs for Download

In [12]:
import shutil

# Zip the outputs/models folder
shutil.make_archive('shopmind_models', 'zip', '.', 'outputs/models')

size_mb = os.path.getsize('shopmind_models.zip') / 1e6
print(f'✓ Created shopmind_models.zip ({size_mb:.0f} MB)')
print('Download this file from JupyterHub file browser → right-click → Download')
print()
print('Files inside:')
for root, dirs, files in os.walk('outputs/models'):
    for f in files:
        fp = os.path.join(root, f)
        print(f'  {fp}  ({os.path.getsize(fp)/1e6:.1f} MB)')

✓ Created shopmind_models.zip (2128 MB)
Download this file from JupyterHub file browser → right-click → Download

Files inside:
  outputs/models/fake_review_model.joblib  (0.3 MB)
  outputs/models/training_meta.json  (0.0 MB)
  outputs/models/absa_model/model.safetensors  (267.8 MB)
  outputs/models/absa_model/config.json  (0.0 MB)
  outputs/models/absa_model/training_args.bin  (0.0 MB)
  outputs/models/absa_model/tokenizer_config.json  (0.0 MB)
  outputs/models/absa_model/vocab.txt  (0.2 MB)
  outputs/models/absa_model/special_tokens_map.json  (0.0 MB)
  outputs/models/absa_model/tokenizer.json  (0.7 MB)
  outputs/models/absa_model/checkpoint-1126/scaler.pt  (0.0 MB)
  outputs/models/absa_model/checkpoint-1126/optimizer.pt  (535.7 MB)
  outputs/models/absa_model/checkpoint-1126/trainer_state.json  (0.0 MB)
  outputs/models/absa_model/checkpoint-1126/model.safetensors  (267.8 MB)
  outputs/models/absa_model/checkpoint-1126/config.json  (0.0 MB)
  outputs/models/absa_model/checkpoint-11